# Vector Databases

Vector databases store **embeddings** — dense numerical representations of text, images, or any data produced by ML models. Unlike SQL (exact match) or MongoDB (field match), they answer the question: *"what items are semantically similar to this query?"*

This is the core primitive behind:
- **RAG** (Retrieval-Augmented Generation) — giving LLMs access to your own documents
- **Semantic search** — search by meaning, not keywords
- **Recommendation systems** — find items similar to what a user liked
- **Duplicate detection** — find near-duplicate records in a dataset

---

## Key Concept: Embeddings

An embedding is a fixed-length vector (e.g. 384 or 1536 numbers) that captures the *meaning* of an input. Two semantically similar inputs will have vectors that are close together in that high-dimensional space.

```
"dog" → [0.12, -0.45, 0.88, ...]   # 384-dim vector
"puppy" → [0.11, -0.43, 0.85, ...]  # very close
"car" → [0.92, 0.31, -0.20, ...]    # far away
```

Similarity is measured with **cosine similarity** or **dot product**.


## Tool Landscape (2026)

| Tool | Best For | Install |
|------|----------|---------|
| **ChromaDB** | Local dev, notebooks, prototyping | `pip install chromadb` |
| **FAISS** | Fast local similarity search (no persistence) | `pip install faiss-cpu` |
| **Pinecone** | Managed cloud, production scale | `pip install pinecone-client` |
| **Weaviate** | Self-hosted, multimodal | Docker or cloud |
| **Qdrant** | High-performance, Rust-based | `pip install qdrant-client` |

We'll use **ChromaDB** here — zero infrastructure, runs in-memory or on disk.


In [ ]:
# pip install chromadb sentence-transformers
import chromadb
from chromadb.utils import embedding_functions

# In-memory client (no persistence) — swap for chromadb.PersistentClient(path='./chroma_db') to save to disk
client = chromadb.Client()

### 1. Create a Collection

A **collection** in ChromaDB is analogous to a table in SQL or a collection in MongoDB — it holds a set of documents and their embeddings.

In [ ]:
# Use a local sentence-transformer model to generate embeddings
# 'all-MiniLM-L6-v2' is small (80MB), fast, and good for English text
embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)

collection = client.create_collection(
    name='ds_notes',
    embedding_function=embed_fn
)
print('Collection created:', collection.name)

### 2. Add Documents

In [ ]:
documents = [
    'Random forests are an ensemble of decision trees trained on random subsets of data.',
    'Gradient boosting builds trees sequentially, each correcting errors of the previous.',
    'A neural network learns hierarchical representations through layers of weighted connections.',
    'K-means clustering assigns points to the nearest centroid and iterates until convergence.',
    'Principal Component Analysis reduces dimensionality by projecting onto axes of maximum variance.',
    'SQL joins combine rows from two or more tables based on a related column.',
    'Pandas DataFrames are two-dimensional labelled data structures with columns of potentially different types.',
]

collection.add(
    documents=documents,
    ids=[f'doc_{i}' for i in range(len(documents))]  # must be unique strings
)

print(f'Added {collection.count()} documents')

### 3. Query by Semantic Similarity

In [ ]:
results = collection.query(
    query_texts=['how do tree-based models work?'],
    n_results=3
)

for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f'Distance: {dist:.4f} | {doc}')

Notice that both the random forest and gradient boosting entries rank above SQL or Pandas — the search is finding *meaning*, not keywords.


---

## FAISS — Fast Local Similarity Search

FAISS (Facebook AI Similarity Search) is a library for efficient nearest-neighbour search. It doesn't manage documents or metadata — just raw vectors. Use it when you need maximum speed or are building your own pipeline.

In [ ]:
# pip install faiss-cpu sentence-transformers
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

# Encode documents to embeddings
embeddings = model.encode(documents, convert_to_numpy=True)
dim = embeddings.shape[1]  # 384 for this model

# Build a flat L2 index (exact search — no approximation)
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

print(f'Index contains {index.ntotal} vectors of dimension {dim}')

In [ ]:
query = model.encode(['dimensionality reduction techniques'], convert_to_numpy=True)

distances, indices = index.search(query, k=3)

for idx, dist in zip(indices[0], distances[0]):
    print(f'L2 dist: {dist:.4f} | {documents[idx]}')

---

## How This Fits into RAG

A basic RAG pipeline looks like:

```
1. Chunk your documents (e.g. 500-token chunks)
2. Embed each chunk → store in vector DB
3. At query time:
   a. Embed the user's question
   b. Retrieve the top-k most similar chunks
   c. Pass chunks + question to an LLM as context
   d. LLM answers grounded in your actual documents
```

See [S9_08_llm_apis.ipynb](../S9_JSON_and_APIs/S9_08_llm_apis.ipynb) for LLM API usage that pairs with this.

---

## Summary

| | SQL | MongoDB | Vector DB |
|---|---|---|---|
| Query type | Exact / range | Field match | Semantic similarity |
| Data | Structured tables | JSON documents | Embeddings + metadata |
| ML use case | Feature stores, logs | Semi-structured data | RAG, semantic search |
| Key tools | SQLite, PostgreSQL | pymongo | ChromaDB, FAISS, Pinecone |
